# 00 — Session Setup

**Run cells top-to-bottom, one at a time. Wait for each to finish before running the next.**

---
### Before running: verify your Colab Secrets

Click the **🔑 key icon** in the left sidebar. You need these secrets with a **blue toggle** (Notebook access ON):
- `ANTHROPIC_API_KEY` — from console.anthropic.com
- `HF_TOKEN` — from huggingface.co/settings/tokens
- `GITHUB_PAT` — from github.com → Settings → Developer settings → Personal access tokens

If any toggle is grey, click it to turn it blue NOW before running anything.

---
### ⚠️ Change your branch name in Cell 3 before running it

In [ ]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, ~15 GB VRAM
# If you see K80 or < 14 GB: Runtime → Disconnect and delete runtime → reconnect
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    icon = '✅' if vram >= 14 else '⚠️ '
    print(f'\n{icon} GPU: {name}  ({vram:.0f} GB VRAM)')
    if vram < 14:
        print('   You have a K80 (12 GB). Reconnect to get a T4 (16 GB).')
else:
    print('\n❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

Tue Sep  1 23:43:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A popup asks for permissions — click Allow on everything (normal Google behaviour).
from google.colab import drive
import os
drive.mount('/content/drive')
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Mount failed — run this cell again')

Mounted at /content/drive
✅ Drive mounted


In [3]:
# ── CELL 3: Clone or pull the GitHub repo ─────────────────────────────────────
#
# ⚠️  CHANGE BRANCH_NAME BEFORE RUNNING
#     Examples:
#       'advisor_colab_experiments'   ← advisor testing
#       'pair-1/smollm2-1.7b'         ← student pair 1
#       'main'                        ← read-only reference (do not push to main)

import os, subprocess, sys
from google.colab import userdata

BRANCH_NAME = 'btt_setup_VD'
REPO_ORG    = 'Break-Through-Tech'
REPO_NAME   = 'Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation'
REPO_DIR    = '/content/project'   # repo root

# ── Load PAT ──────────────────────────────────────────────────────────────────
try:
    PAT = userdata.get('GITHUB_PAT')
    assert PAT, 'Secret is empty'
    print(f'✅ GITHUB_PAT loaded ({len(PAT)} chars)')
except Exception as e:
    print(f'❌ GITHUB_PAT: {e}')
    print('   Open 🔑 Secrets → add GITHUB_PAT → toggle Notebook access ON')
    raise SystemExit('Cannot clone without GITHUB_PAT')

REPO_URL = f'https://{PAT}@github.com/{REPO_ORG}/{REPO_NAME}.git'

def git(args, cwd=None, check=True):
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(f'❌ git error: {r.stderr.replace(PAT, "***").strip()}')
        raise RuntimeError(' '.join(args))
    return r.stdout.strip()

# ── Clone or pull ──────────────────────────────────────────────────────────────
if not os.path.exists(f'{REPO_DIR}/.git'):
    if os.path.exists(REPO_DIR):
        print('Removing broken directory ...')
        subprocess.run(['rm', '-rf', REPO_DIR])
    print(f'Cloning branch "{BRANCH_NAME}" ...')
    git(['git', 'clone', '-b', BRANCH_NAME, REPO_URL, REPO_DIR])
    print(f'✅ Cloned to {REPO_DIR}')
else:
    print(f'Pulling latest from "{BRANCH_NAME}" ...')
    git(['git', 'checkout', BRANCH_NAME], cwd=REPO_DIR)
    git(['git', 'pull', 'origin', BRANCH_NAME], cwd=REPO_DIR)
    print(f'✅ Up to date')

# ── Auto-detect where main.py lives (repo root or code/ subfolder) ────────────
if os.path.exists(f'{REPO_DIR}/main.py'):
    CODE_DIR = REPO_DIR
elif os.path.exists(f'{REPO_DIR}/code/main.py'):
    CODE_DIR = f'{REPO_DIR}/code'
else:
    CODE_DIR = None
    print('❌ Cannot find main.py — checked repo root and code/ subfolder')
    print(f'   Contents of {REPO_DIR}: {os.listdir(REPO_DIR)}')

if CODE_DIR:
    print(f'✅ Code directory: {CODE_DIR}')
    missing = [f for f in ['requirements.txt', 'requirements_colab.txt']
               if not os.path.exists(f'{CODE_DIR}/{f}')]
    if missing:
        print(f'⚠️  Missing in {CODE_DIR}: {missing}')
        print('   Push these files from your local machine, or run:')
        print('   !git -C /content/project fetch origin')
        print('   !git -C /content/project checkout origin/main -- code/requirements_colab.txt')
    else:
        print('✅ requirements.txt and requirements_colab.txt found')

    os.chdir(CODE_DIR)
    sys.path.insert(0, CODE_DIR)
    # Store CODE_DIR for other cells to use
    os.environ['SLM_CODE_DIR'] = CODE_DIR
    print(f'Working directory: {os.getcwd()}')

✅ GITHUB_PAT loaded (40 chars)
Cloning branch "btt_setup_VD" ...
✅ Cloned to /content/project
✅ Code directory: /content/project/code
✅ requirements.txt and requirements_colab.txt found
Working directory: /content/project/code


In [ ]:
# ── CELL 4: Install dependencies ──────────────────────────────────────────────
# Reads requirements files from the code directory found in Cell 3.
# Takes 3–4 minutes. Normal to see some warnings.
import os, subprocess, sys

CODE_DIR = os.environ.get('SLM_CODE_DIR', '/content/project/code')
print(f'Installing from: {CODE_DIR}')

def pip_install(filename):
    path = f'{CODE_DIR}/{filename}'
    if not os.path.exists(path):
        print(f'❌ {filename} not found at {path}')
        print('   Make sure Cell 3 ran successfully first.')
        return False
    print(f'\nInstalling {filename} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', path],
        capture_output=True, text=True
    )
    tail = (result.stdout + result.stderr).strip().split('\n')
    for line in tail[-5:]:
        if line.strip():
            print(f'  {line}')
    if result.returncode != 0:
        print(f'❌ pip failed for {filename}')
        return False
    print(f'✅ {filename} done')
    return True

ok1 = pip_install('requirements.txt')
ok2 = pip_install('requirements_colab.txt')
print('\n✅ All dependencies installed' if (ok1 and ok2) else '\n⚠️  Check errors above')

Installing from: /content/project/code

Installing requirements.txt ...
✅ requirements.txt done

Installing requirements_colab.txt ...
✅ requirements_colab.txt done

✅ All dependencies installed


In [ ]:
# ── CELL 5: Drive folder structure and HuggingFace cache ─────────────────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
for d in [f'{DRIVE_ROOT}/data/raw', f'{DRIVE_ROOT}/data/processed',
          f'{DRIVE_ROOT}/data/checkpoints', f'{DRIVE_ROOT}/outputs',
          f'{DRIVE_ROOT}/hf_cache']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['DRIVE_ROOT'] = DRIVE_ROOT

print(f'✅ Drive folders ready under {DRIVE_ROOT}')
print(f'✅ HF model cache → {os.environ["HF_HOME"]}')

✅ Drive folders ready under /content/drive/MyDrive/slm-distillation
✅ HF model cache → /content/drive/MyDrive/slm-distillation/hf_cache


In [ ]:
# ── CELL 6: Load API keys from Colab Secrets ──────────────────────────────────
import os
from google.colab import userdata

def load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'  ✅ {name}')
            return True
        print(f'  ⚠️  {name} is empty — {"add value in 🔑 Secrets" if required else "optional"}')
    except Exception:
        print(f'  ❌ {name} not found — {"open 🔑 Secrets and toggle Notebook access ON" if required else "optional"}')
    return not required

print('Loading secrets:')
all_ok = all([
    load_secret('ANTHROPIC_API_KEY', required=True),
    load_secret('HF_TOKEN',          required=True),
    load_secret('OPENAI_API_KEY',    required=False),
])
print('\n✅ Required secrets loaded' if all_ok else '\n❌ Fix missing secrets before running the pipeline')

Loading secrets:
  ✅ ANTHROPIC_API_KEY
  ✅ HF_TOKEN
  ❌ OPENAI_API_KEY not found — optional

✅ Required secrets loaded


In [ ]:
# ── CELL 7: Verify everything is ready ───────────────────────────────────────
import os, torch

code_dir  = os.environ.get('SLM_CODE_DIR', '')
drive_ok  = os.path.exists('/content/drive/MyDrive')
gpu_ok    = torch.cuda.is_available()
vram_ok   = gpu_ok and torch.cuda.get_device_properties(0).total_memory > 14e9

checks = [
    ('T4 GPU (≥14 GB VRAM)',   vram_ok),
    ('Drive mounted',          drive_ok),
    ('Code directory found',   bool(code_dir) and os.path.exists(code_dir)),
    ('main.py present',        os.path.exists(f'{code_dir}/main.py') if code_dir else False),
    ('ANTHROPIC_API_KEY set',  'ANTHROPIC_API_KEY' in os.environ),
    ('HF_TOKEN set',           'HF_TOKEN' in os.environ),
    ('HF cache on Drive',      os.environ.get('HF_HOME','').startswith('/content/drive')),
]

print('Setup verification:')
print(f'  Code directory: {code_dir or "NOT SET"}')
print()
all_ok = True
for label, ok in checks:
    print(f'  {"✅" if ok else "❌"} {label}')
    if not ok:
        all_ok = False

print()
print('🚀 Ready! Scroll down to run the pipeline.' if all_ok else
      '⚠️  Fix ❌ items before running the pipeline.')

Setup verification:
  Code directory: /content/project/code

  ✅ T4 GPU (≥14 GB VRAM)
  ✅ Drive mounted
  ✅ Code directory found
  ✅ main.py present
  ✅ ANTHROPIC_API_KEY set
  ✅ HF_TOKEN set
  ✅ HF cache on Drive

🚀 Ready! Scroll down to run the pipeline.


---
## Run the pipeline

Choose one of the cells below. The `$SLM_CODE_DIR` variable is set by Cell 3.

In [ ]:
# ── Full training + evaluation run (~40–60 min on T4) ─────────────────────────
import os
CODE = os.environ.get('SLM_CODE_DIR', '/content/project/code')
!python "$CODE/main.py" \
    --phase 1 \
    --config "$CODE/configs/phase1_config.yaml" \
    --device_mode colab

23:46:16 | INFO     | __main__ | [main] Overriding device_mode: local_mps → colab
23:46:16 | INFO     | __main__ | [main] Colab mode: drive_root → /content/drive/MyDrive/slm-distillation
23:46:16 | INFO     | __main__ | [main] Drive already mounted — skipping drive.mount().
23:46:16 | INFO     | __main__ | [main] HF cache → /content/drive/MyDrive/slm-distillation/hf_cache
23:46:16 | INFO     | __main__ | [main] Colab Secrets loaded.
23:46:16 | INFO     | __main__ | 
  SLM Distillation — Phase 1 | Mode: train
  Config:      /content/project/code/configs/phase1_config.yaml
  Device mode: colab
  Student SLM: HuggingFaceTB/SmolLM2-360M-Instruct
  Teacher LLM: claude-haiku-4-5
  Drive root:  /content/drive/MyDrive/slm-distillation
23:46:16 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.
23:46:18 | INFO     | phase1.pipeline | [pipeline] Run ID : 20260901_2346_SmolLM2-360M-Instruct_ep3
23:46:18 | INFO     | phase1.pipeline | [pipeline] Outputs → /content/drive/MyDrive/slm-dist

In [ ]:
# ── Live demo mode ────────────────────────────────────────────────────────────
import os
CODE  = os.environ.get('SLM_CODE_DIR', '/content/project/code')
DRIVE = os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/slm-distillation')

ADAPTER = f'{DRIVE}/outputs/20260901_2346_SmolLM2-360M-Instruct_ep3/models/lora_adapter'  # ← UPDATE

!python "$CODE/main.py" \
    --phase 1 \
    --config "$CODE/configs/phase1_config.yaml" \
    --device_mode colab \
    --mode demo \
    --adapter_dir "$ADAPTER"

23:54:39 | INFO     | __main__ | [main] Overriding device_mode: local_mps → colab
23:54:39 | INFO     | __main__ | [main] Colab mode: drive_root → /content/drive/MyDrive/slm-distillation
23:54:40 | INFO     | __main__ | [main] Drive already mounted — skipping drive.mount().
23:54:40 | INFO     | __main__ | [main] HF cache → /content/drive/MyDrive/slm-distillation/hf_cache
23:54:40 | INFO     | __main__ | [main] Colab Secrets loaded.
23:54:40 | INFO     | __main__ | 
  SLM Distillation — Phase 1 | Mode: demo
  Config:      /content/project/code/configs/phase1_config.yaml
  Device mode: colab
  Student SLM: HuggingFaceTB/SmolLM2-360M-Instruct
  Teacher LLM: claude-haiku-4-5
  Drive root:  /content/drive/MyDrive/slm-distillation
23:54:40 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.
23:54:48 | WARNING  | torchao | Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torch

In [ ]:
# ── Push code changes to GitHub ───────────────────────────────────────────────
import subprocess, os

MSG = 'describe your changes'   # ← UPDATE
REPO = '/content/project'

for cmd in [['git','add','.'], ['git','commit','-m',MSG], ['git','push']]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
print('Done')

---
## First session only — create your branch

Only needed if your branch does not yet exist on GitHub.

In [ ]:
# ── Create branch on GitHub (run ONCE) ───────────────────────────────────────
import subprocess

MY_BRANCH = 'pair-X/model-name'   # ← CHANGE THIS

for cmd in [
    ['git', 'checkout', '-b', MY_BRANCH],
    ['git', 'push', '-u', 'origin', MY_BRANCH],
]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/project')
    print((r.stdout + r.stderr).strip())

print(f"\n✅ Branch '{MY_BRANCH}' created.")
print(f"Change BRANCH_NAME in Cell 3 to '{MY_BRANCH}' for all future sessions.")

Commit

In [5]:
import subprocess

# search common Colab save locations
r = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', '*.ipynb', '-newermt', '-2 hours'],
    capture_output=True, text=True
)
print("Recently modified notebooks in Drive:\n", r.stdout)

Recently modified notebooks in Drive:
 /content/drive/MyDrive/Colab Notebooks/SmolLM2-360M-Instruct_ep3.ipynb
/content/drive/MyDrive/Colab Notebooks/dataset_cleaning_VD.ipynb



In [6]:
import subprocess
import os
import shutil

# ── EDIT THESE TWO LINES ──────────────────────────────────────────
SOURCE_PATH = '/content/drive/MyDrive/Colab Notebooks/SmolLM2-360M-Instruct.ipynb'  # ← from Step 1
NOTEBOOK_NAME = 'smollm2_360m_model.ipynb'  # ← what you want it called in the repo
# ───────────────────────────────────────────────────────────────────

REPO = '/content/project'
BRANCH = 'btt_setup_VD'
DEST_PATH = f'{REPO}/notebooks/{NOTEBOOK_NAME}'

# make sure notebooks/ folder exists
os.makedirs(f'{REPO}/notebooks', exist_ok=True)

# copy the file in
shutil.copy(SOURCE_PATH, DEST_PATH)
print(f"Copied to: {DEST_PATH}")

# set git identity (safe to re-run even if already set)
subprocess.run(['git', 'config', 'user.email', 'your-actual-email@domain.com'], cwd=REPO)
subprocess.run(['git', 'config', 'user.name', 'Your Full Name'], cwd=REPO)

# commit and push
for cmd in [
    ['git', 'checkout', BRANCH],
    ['git', 'add', f'notebooks/{NOTEBOOK_NAME}'],
    ['git', 'commit', '-m', f'Add {NOTEBOOK_NAME}'],
    ['git', 'push', 'origin', BRANCH],
]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)

print('\nDone. Verify at:')
print(f'https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation/tree/{BRANCH}/notebooks')

Copied to: /content/project/notebooks/smollm2_360m_model.ipynb
Your branch is up to date with 'origin/btt_setup_VD'.
Already on 'btt_setup_VD'
[btt_setup_VD c7e1aaf] Add smollm2_360m_model.ipynb
 1 file changed, 1 insertion(+)
 create mode 100644 notebooks/smollm2_360m_model.ipynb
To https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git
   ec40fa7..c7e1aaf  btt_setup_VD -> btt_setup_VD

Done. Verify at:
https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation/tree/btt_setup_VD/notebooks
